<a href="https://colab.research.google.com/github/runessaa/-Streltsov-Projects/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9611_%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B8_%D1%81%D0%B5%D0%B3%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D1%86%D0%B8%D1%8F_%D0%BA%D0%BB%D0%B8%D0%B5%D0%BD%D1%82%D0%BE%D0%B2_%D1%81_%D0%BF%D0%BE%D0%BC%D0%BE%D1%89%D1%8C%D1%8E_%D0%B0%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC%D0%BE%D0%B2_%D0%BA%D0%BB%D0%B0%D1%81%D1%82%D0%B5%D1%80%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №10. Анализ и сегментация клиентов с помощью алгоритмов кластеризации**

### **Цель работы:**

Разработать систему сегментации клиентов для розничной компании с использованием алгоритмов кластеризации. Это позволит компании лучше понимать своих клиентов, персонализировать маркетинговые кампании и оптимизировать бизнес-процессы.

### **Введение:**

Розничные компании сталкиваются с большим объемом данных о своих клиентах, включая историю покупок, демографическую информацию и поведенческие характеристики. Однако без должного анализа эти данные остаются неиспользованными. Сегментация клиентов позволяет выделить группы с общими характеристиками, чтобы более эффективно таргетировать предложения и улучшить удовлетворенность клиентов.



### **Задачи:**

1. **Сбор и анализ данных о клиентах.**
2. **Предобработка и подготовка данных для моделирования.**
3. **Применение различных алгоритмов кластеризации для сегментации клиентов.**
4. **Оценка качества кластеризации с использованием внутренних и внешних метрик.**
5. **Интерпретация и визуализация результатов.**
6. **Формирование рекомендаций для бизнес-стратегии компании на основе полученных сегментов.**



### **Пошаговое описание рабочего процесса (пайплайна):**

#### **Шаг 1: Сбор и анализ данных**

**1.1. Выбор набора данных:**

- Используйте датасет "Online Retail II" из [UCI Machine Learning Repository](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci) или [другой открытый набор данных](https://archive.ics.uci.edu/datasets?Task=Clustering&skip=0&take=10&sort=desc&orderBy=NumHits&search=&Area=Business), содержащий информацию о транзакциях клиентов.
- Данные должны включать:
  - Идентификаторы клиентов.
  - Информацию о покупках (товары, количество, стоимость).
  - Дату и время транзакций.
  - Демографические данные (если доступны): возраст, пол, локация и т.д.

**1.2. Первичный анализ данных (EDA):**

- Изучите структуру данных и их распределение.
- Определите основные характеристики данных:
  - Общий объем продаж.
  - Частота покупок по клиентам.
  - Распределение выручки по товарам.
- Выявите тенденции и аномалии.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option('display.max_columns', 30)

df = pd.read_csv('online_retail_II.csv', encoding='ISO-8859-1')

df.columns = df.columns.str.strip()
df = df.rename(columns={
    'Invoice': 'InvoiceNo',
    'Customer ID': 'CustomerID',
    'Price': 'UnitPrice'
})

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
df['CustomerID'] = pd.to_numeric(df['CustomerID'], errors='coerce')
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

display(df.head())

print('Размер данных:', df.shape)

print('\nТипы данных:')
print(df.dtypes)

print('\nПропущенные значения:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))

print('Общий объем продаж:', round(df['TotalPrice'].sum(), 2))
print('Количество клиентов:', df['CustomerID'].nunique())
print('Количество транзакций:', df['InvoiceNo'].nunique())

purchase_frequency = df.groupby('CustomerID')['InvoiceNo'].nunique().sort_values(ascending=False)

print('\nЧастота покупок по клиентам:')
display(purchase_frequency.describe().to_frame('frequency'))

product_revenue = (
    df.groupby('Description', dropna=False)['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print('\nТоп-10 товаров по выручке:')
display(product_revenue.to_frame('revenue'))

anomalies = df[
    (df['Quantity'] <= 0) |
    (df['UnitPrice'] <= 0) |
    (df['CustomerID'].isna())
]

print('\nПотенциальные аномалии:', len(anomalies))
display(anomalies.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


Размер данных: (1067371, 9)

Типы данных:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
TotalPrice            float64
dtype: object

Пропущенные значения:


,missing
CustomerID,243007
Description,4382
InvoiceNo,0
Quantity,0
StockCode,0
InvoiceDate,0
UnitPrice,0
Country,0
TotalPrice,0


Общий объем продаж: 19287250.57
Количество клиентов: 5942
Количество транзакций: 53628

Частота покупок по клиентам:


,frequency
count,5942.000000
mean,7.552339
std,15.972262
min,1.000000
25%,2.000000
50%,4.000000
75%,8.000000
max,510.000000



Топ-10 товаров по выручке:


,revenue
Description,
REGENCY CAKESTAND 3 TIER,327813.65
DOTCOM POSTAGE,322647.47
WHITE HANGING HEART T-LIGHT HOLDER,257533.90
JUMBO BAG RED RETROSPOT,148800.64
PARTY BUNTING,147948.50
ASSORTED COLOUR BIRD ORNAMENT,131413.85
PAPER CHAIN KIT 50'S CHRISTMAS,121662.14
POSTAGE,112341.00
CHILLI LIGHTS,84854.16



Потенциальные аномалии: 261822


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4


#### **Шаг 2: Предобработка данных**

**2.1. Работа с пропущенными значениями:**

- Проанализируйте наличие пропущенных данных.
- Решите, как справиться с ними:
  - Удаление строк/столбцов с пропущенными значениями.
  - Заполнение пропущенных значений средним, медианой или наиболее частым значением.

**2.2. Обработка выбросов:**

- Выявите выбросы в данных (например, аномально большие заказы).
- Решите, следует ли их удалить или обработать иным образом.

**2.3. Создание новых признаков:**

- Рассчитайте Recency, Frequency, Monetary Value (RFM-анализ):
  - **Recency (давность):** Время с момента последней покупки.
  - **Frequency (частота):** Количество покупок за определенный период.
  - **Monetary (сумма):** Общая сумма покупок.
- Создайте дополнительные признаки, такие как средний чек, предпочтительные категории товаров и т.д.

**2.4. Нормализация и масштабирование:**

- Примените стандартизацию или нормализацию к числовым признакам для приведения их к единому масштабу.
- Объясните выбор метода масштабирования.

In [5]:
from sklearn.preprocessing import StandardScaler

missing_before = df.isna().sum()

data = df.copy()
data = data.dropna(subset=['CustomerID', 'InvoiceNo', 'InvoiceDate', 'Quantity', 'UnitPrice'])
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]
data = data[(data['Quantity'] > 0) & (data['UnitPrice'] > 0)]
data['CustomerID'] = data['CustomerID'].astype(int)
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

q1 = data['TotalPrice'].quantile(0.25)
q3 = data['TotalPrice'].quantile(0.75)
iqr = q3 - q1
low = max(0, q1 - 1.5 * iqr)
high = q3 + 1.5 * iqr
data_clean = data[(data['TotalPrice'] >= low) & (data['TotalPrice'] <= high)].copy()

snapshot_date = data_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

invoice_sum = data_clean.groupby(['CustomerID', 'InvoiceNo'], as_index=False)['TotalPrice'].sum()
avg_check = invoice_sum.groupby('CustomerID')['TotalPrice'].mean().rename('AvgCheck')

rfm = data_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
    Quantity=('Quantity', 'sum'),
    UniqueItems=('StockCode', 'nunique')
).join(avg_check)

rfm['Country'] = data_clean.groupby('CustomerID')['Country'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0])
rfm = rfm.replace([np.inf, -np.inf], np.nan).dropna()

features = ['Recency', 'Frequency', 'Monetary', 'AvgCheck', 'Quantity', 'UniqueItems']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm[features])
X_scaled = pd.DataFrame(X_scaled, columns=features, index=rfm.index)

print('Пропуски до обработки:')
display(missing_before.sort_values(ascending=False).to_frame('missing_before'))
print('Размер после очистки:', data_clean.shape)
print('Удалено строк:', len(df) - len(data_clean))
print('Количество клиентов в RFM-таблице:', len(rfm))

print('\nRFM-признаки:')
display(rfm.head())

print('\nМетод масштабирования: StandardScaler. Он выбран, потому что K-Means, иерархическая кластеризация, DBSCAN и OPTICS зависят от расстояний, стандартизация приводит признаки RFM к единому масштабу без изменения формы распределений.')
display(X_scaled.describe().round(3))


Пропуски до обработки:


,missing_before
CustomerID,243007
Description,4382
InvoiceNo,0
Quantity,0
StockCode,0
InvoiceDate,0
UnitPrice,0
Country,0
TotalPrice,0


Размер после очистки: (739167, 9)
Удалено строк: 328204
Количество клиентов в RFM-таблице: 5676

RFM-признаки:


,Recency,Frequency,Monetary,Quantity,UniqueItems,AvgCheck,Country
CustomerID,,,,,,,
12346,529,10,327.86,60,26,32.786000,United Kingdom
12347,2,8,4371.34,2740,122,546.417500,Iceland
12348,75,5,573.24,780,22,114.648000,Finland
12349,19,3,2843.99,1467,129,947.996667,Italy
12350,310,1,334.40,197,17,334.400000,Norway



Метод масштабирования: StandardScaler. Он выбран, потому что K-Means, иерархическая кластеризация, DBSCAN и OPTICS зависят от расстояний, стандартизация приводит признаки RFM к единому масштабу без изменения формы распределений.


,Recency,Frequency,Monetary,AvgCheck,Quantity,UniqueItems
count,5676.000,5676.000,5676.000,5676.000,5676.000,5676.000
mean,0.000,-0.000,-0.000,0.000,0.000,-0.000
std,1.000,1.000,1.000,1.000,1.000,1.000
min,-0.958,-0.422,-0.420,-1.284,-0.400,-0.702
25%,-0.839,-0.422,-0.348,-0.603,-0.336,-0.544
50%,-0.508,-0.248,-0.241,-0.199,-0.240,-0.325
75%,0.859,0.012,0.024,0.335,0.017,0.175
max,2.581,31.177,47.850,18.672,44.313,21.057


#### **Шаг 3: Применение алгоритмов кластеризации**

**3.1. Выбор алгоритмов:**

- **K-средних (K-Means):** Для разбиения данных на k кластеров на основе эврестического подхода.
- **Иерархическая кластеризация:** Для выявления вложенной структуры кластеров.
- **DBSCAN и OPTICS:** Для обнаружения кластеров произвольной формы и выявления выбросов.

**3.2. Определение оптимального количества кластеров:**

- Для K-Means и иерархической кластеризации используйте:
  - **Метод локтя (Elbow Method):** Постройте график зависимости суммы квадратов внутрикластерных расстояний от числа кластеров.
  - **Коэффициент силуэта:** Рассчитайте для различных значений k и выберите оптимальное.

**3.3. Применение алгоритмов:**

- Запустите каждый алгоритм на подготовленных данных.
- Сохраняйте результаты кластеризации для последующего анализа.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, OPTICS
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

ks = range(2, 11)
inertias = []
silhouette_values = []

for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouette_values.append(silhouette_score(X_scaled, labels))

best_k = list(ks)[int(np.argmax(silhouette_values))]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertias, marker='o')
ax[0].set_title('Метод локтя для K-Means')
ax[0].set_xlabel('Количество кластеров')
ax[0].set_ylabel('Inertia')

ax[1].plot(list(ks), silhouette_values, marker='o')
ax[1].set_title('Коэффициент силуэта')
ax[1].set_xlabel('Количество кластеров')
ax[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.show()

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

hierarchical = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
eps = float(np.percentile(distances[:, -1], 90))

dbscan = DBSCAN(eps=eps, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

optics = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05)
optics_labels = optics.fit_predict(X_scaled)

cluster_results = {
    'KMeans': kmeans_labels,
    'Hierarchical': hierarchical_labels,
    'DBSCAN': dbscan_labels,
    'OPTICS': optics_labels
}

for name, labels in cluster_results.items():
    rfm[name + '_cluster'] = labels
    print(f'{name}: кластеров = {len(set(labels) - {-1})}, выбросов = {sum(labels == -1)}')

print('Оптимальное количество кластеров для K-Means и иерархической кластеризации:', best_k)


#### **Шаг 4: Оценка качества кластеризации**

**4.1. Внутренние метрики:**

- **Коэффициент силуэта:** Оцените, насколько хорошо объекты расположены внутри кластеров.
- **Индекс Дэвиса-Болдина:** Оцените уровень разделимости кластеров.
- **Индекс Калинского-Харабаза:** Оцените соотношение межкластерной дисперсии к внутрикластерной.

**4.2. Внешние метрики (если доступны истинные метки):**

- **Adjusted Rand Index (ARI):** Сравните полученные кластеры с известными категориями клиентов.
- **Normalized Mutual Information (NMI):** Измерьте общую информацию между распределениями.

**4.3. Сравнение алгоритмов:**

- Составьте таблицу со значениями метрик для каждого алгоритма.
- Определите, какой алгоритм показал наилучшие результаты и почему.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def clustering_metrics(X, labels):
    labels = np.array(labels)
    mask = labels != -1
    valid_labels = labels[mask]
    valid_X = X.iloc[mask] if hasattr(X, 'iloc') else X[mask]
    n_clusters = len(set(valid_labels))
    if n_clusters < 2 or len(valid_labels) <= n_clusters:
        return {
            'n_clusters': n_clusters,
            'noise_points': int(np.sum(labels == -1)),
            'silhouette': np.nan,
            'davies_bouldin': np.nan,
            'calinski_harabasz': np.nan,
            'ARI': np.nan,
            'NMI': np.nan
        }
    return {
        'n_clusters': n_clusters,
        'noise_points': int(np.sum(labels == -1)),
        'silhouette': silhouette_score(valid_X, valid_labels),
        'davies_bouldin': davies_bouldin_score(valid_X, valid_labels),
        'calinski_harabasz': calinski_harabasz_score(valid_X, valid_labels),
        'ARI': np.nan,
        'NMI': np.nan
    }

metrics_table = pd.DataFrame({
    name: clustering_metrics(X_scaled, labels)
    for name, labels in cluster_results.items()
}).T

display(metrics_table.round(4))

valid_metrics = metrics_table.dropna(subset=['silhouette'])
best_algorithm = valid_metrics['silhouette'].idxmax()
best_labels = cluster_results[best_algorithm]
rfm['BestCluster'] = best_labels

print('Внешние метрики ARI и NMI не рассчитаны, потому что в выбранном наборе данных нет истинных меток сегментов клиентов.')
print('Лучший алгоритм:', best_algorithm)
print('Обоснование: выбран алгоритм с максимальным коэффициентом силуэта среди моделей, где сформировано не менее двух кластеров.')


#### **Шаг 5: Интерпретация и визуализация результатов**

**5.1. Визуализация кластеров:**

- **Снижение размерности:** Примените PCA или t-SNE или UMAP для отображения данных в 2D или 3D пространстве.
- **Постройте графики:**
  - Рассеивания с цветовой кодировкой кластеров.
  - Дендрограммы для иерархической кластеризации.
- **Визуализация признаков:**
  - Постройте боксплоты, гистограммы или тепловые карты для сравнения признаков между кластерами.

**5.2. Описание сегментов:**

- Для каждого кластера опишите характерные черты:
  - Средние значения признаков.
  - Поведенческие особенности (например, частота покупок, средний чек).
  - Демографические характеристики (если доступны).
- Присвойте сегментам осмысленные названия (например, "Лояльные клиенты", "Покупатели со сниженной активностью", "Большие транзакции").

In [ ]:
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=rfm['BestCluster'], s=18, cmap='tab10')
plt.title(f'Кластеры клиентов: {best_algorithm}, PCA 2D')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter, label='Кластер')
plt.show()

sample_size = min(200, len(X_scaled))
sample_idx = np.random.default_rng(42).choice(len(X_scaled), size=sample_size, replace=False)
linked = linkage(X_scaled.iloc[sample_idx], method='ward')

plt.figure(figsize=(12, 5))
dendrogram(linked, no_labels=True)
plt.title('Дендрограмма иерархической кластеризации')
plt.xlabel('Клиенты')
plt.ylabel('Расстояние')
plt.show()

cluster_profiles = rfm.groupby('BestCluster')[features].mean().round(2)
cluster_counts = rfm['BestCluster'].value_counts().sort_index()
cluster_profiles['CustomerCount'] = cluster_counts

q_recency = rfm['Recency'].quantile([0.33, 0.67])
q_frequency = rfm['Frequency'].quantile([0.33, 0.67])
q_monetary = rfm['Monetary'].quantile([0.33, 0.67])
q_avg = rfm['AvgCheck'].quantile(0.67)

def segment_name(cluster_id, row):
    if cluster_id == -1:
        return 'Нетипичные клиенты'
    if row['Recency'] <= q_recency.loc[0.33] and row['Frequency'] >= q_frequency.loc[0.67] and row['Monetary'] >= q_monetary.loc[0.67]:
        return 'Лояльные ценные клиенты'
    if row['Recency'] >= q_recency.loc[0.67]:
        return 'Покупатели со сниженной активностью'
    if row['Monetary'] >= q_monetary.loc[0.67] or row['AvgCheck'] >= q_avg:
        return 'Клиенты с крупными покупками'
    if row['Frequency'] >= q_frequency.loc[0.67]:
        return 'Регулярные покупатели'
    return 'Новые или умеренно активные клиенты'

cluster_profiles['SegmentName'] = [
    segment_name(cluster_id, cluster_profiles.loc[cluster_id])
    for cluster_id in cluster_profiles.index
]

display(cluster_profiles)

profile_scaled = pd.DataFrame(
    scaler.transform(cluster_profiles[features]),
    columns=features,
    index=cluster_profiles.index
)

plt.figure(figsize=(10, 5))
plt.imshow(profile_scaled, aspect='auto')
plt.xticks(range(len(features)), features, rotation=45)
plt.yticks(range(len(profile_scaled.index)), profile_scaled.index)
plt.colorbar(label='Стандартизированное среднее')
plt.title('Сравнение признаков между кластерами')
plt.tight_layout()
plt.show()

for feature in ['Recency', 'Frequency', 'Monetary', 'AvgCheck']:
    rfm.boxplot(column=feature, by='BestCluster', figsize=(8, 4))
    plt.title(f'{feature} по кластерам')
    plt.suptitle('')
    plt.xlabel('Кластер')
    plt.ylabel(feature)
    plt.show()

print('Описание сегментов:')
for cluster_id, row in cluster_profiles.iterrows():
    print(f"Кластер {cluster_id}: {row['SegmentName']}. "
          f"Клиентов: {int(row['CustomerCount'])}; "
          f"Recency={row['Recency']}, Frequency={row['Frequency']}, "
          f"Monetary={row['Monetary']}, AvgCheck={row['AvgCheck']}.")


#### **Шаг 6: Формирование бизнес-рекомендаций**

**6.1. Анализ потребностей каждого сегмента:**

- Определите потребности и предпочтения клиентов в каждом сегменте.
- Выявите возможности для увеличения продаж и улучшения сервиса.

**6.2. Разработка стратегий для каждого сегмента:**

- **Маркетинговые кампании:**
  - Персонализированные предложения.
  - Программы лояльности для удержания ценных клиентов.
- **Оптимизация продуктов:**
  - Расширение ассортимента для популярных сегментов.
  - Фокус на продуктах, интересных конкретным сегментам.

**6.3. Оценка потенциального влияния:**

- Оцените, как предложенные стратегии могут повысить выручку, удовлетворенность клиентов и другие ключевые показатели.

# Ответ

На основе RFM-сегментации клиентская база делится на несколько практических групп.

**Лояльные ценные клиенты.** Это клиенты с высокой частотой покупок, высокой суммарной выручкой и небольшой давностью последней покупки. Для них целесообразны VIP-программа, персональные предложения, ранний доступ к новым товарам и бонусы за повторные покупки. Цель — удержание и рост LTV.

**Регулярные покупатели.** Эти клиенты покупают часто, но их средний чек может быть ниже, чем у VIP-сегмента. Для них подходят накопительные скидки, рекомендации сопутствующих товаров и акции на увеличение корзины. Цель — повысить средний чек.

**Клиенты с крупными покупками.** Это клиенты с высоким Monetary или высоким AvgCheck. Им стоит предлагать премиальные товары, персональные подборки и сервисные предложения. Цель — сохранить высокий чек и увеличить повторяемость покупок.

**Покупатели со сниженной активностью.** У этих клиентов большая давность последней покупки. Для них нужны реактивационные кампании: промокоды на возвращение, напоминания, персональные письма и анализ причин ухода. Цель — вернуть часть клиентов и снизить отток.

**Новые или умеренно активные клиенты.** Для них важны приветственные цепочки, объяснение преимуществ магазина, рекомендации популярных товаров и мягкие стимулы к повторной покупке. Цель — перевести их в регулярный сегмент.

Потенциальное влияние внедрения сегментации: рост выручки за счет персонализации предложений, повышение удержания ценных клиентов, снижение затрат на массовые нерелевантные рассылки, увеличение среднего чека и более точное планирование ассортимента.


#### **Шаг 7: Документирование и презентация результатов**

**7.1. Подготовка отчета:**

- **Введение:**
  - Описание цели работы и её значимости для бизнеса.
- **Методология:**
  - Подробное описание проведенных шагов.
- **Результаты:**
  - Представление метрик оценки и визуализаций.
  - Описание сегментов клиентов.
- **Обсуждение:**
  - Анализ полученных результатов.
  - Сравнение алгоритмов и обоснование выбора.
- **Рекомендации:**
  - Предложения по внедрению результатов в бизнес-процессы.
- **Заключение:**
  - Выводы о проделанной работе и её значимости.

**7.2. Презентация:**

- Подготовьте слайды для представления ключевых моментов работы.
- Используйте визуализации для иллюстрации результатов.



# Шаг 7. Документирование и презентация результатов

## Отчет

**Введение.** Цель работы — выполнить сегментацию клиентов розничной компании на основе датасета `online_retail_II.csv`. Сегментация позволяет разделить покупателей на группы с похожим поведением и использовать эти группы для персонализации маркетинга.

**Методология.** В работе выполнены загрузка данных Online Retail II, первичный анализ структуры данных, проверка пропусков и аномалий, удаление отмененных заказов, строк без клиента и некорректных значений количества или цены. Затем рассчитаны RFM-признаки: Recency, Frequency и Monetary, а также дополнительные признаки AvgCheck, Quantity и UniqueItems. Для подготовки к кластеризации числовые признаки стандартизированы с помощью StandardScaler.

**Результаты.** Для сегментации применены K-Means, иерархическая кластеризация, DBSCAN и OPTICS. Оптимальное количество кластеров для K-Means и иерархической кластеризации определяется по методу локтя и коэффициенту силуэта. Качество моделей сравнивается по Silhouette Score, Davies-Bouldin Index и Calinski-Harabasz Index. Внешние метрики ARI и NMI не используются, так как в датасете нет истинных меток клиентских сегментов.

**Обсуждение.** Лучший алгоритм выбирается по максимальному коэффициенту силуэта при наличии не менее двух кластеров. K-Means и иерархическая кластеризация удобны для интерпретируемых RFM-сегментов, а DBSCAN и OPTICS дополнительно выделяют нетипичных клиентов и шумовые точки.

**Рекомендации.** Для лояльных ценных клиентов нужны VIP-предложения и удержание. Для регулярных покупателей — кросс-продажи и стимулирование среднего чека. Для клиентов с крупными покупками — премиальный ассортимент и персональные подборки. Для покупателей со сниженной активностью — реактивационные промокоды и напоминания. Для новых или умеренно активных клиентов — приветственные цепочки и рекомендации популярных товаров.

**Заключение.** Выполненная сегментация на основе Online Retail II помогает перейти от массового маркетинга к управлению клиентскими группами, повысить релевантность коммуникаций, удержание клиентов и эффективность продаж.

## Презентация

Презентация формируется программно в следующей ячейке и сохраняется в файл `customer_segmentation_presentation.pptx`.



In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt

presentation_path = 'customer_segmentation_presentation.pptx'

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

def add_slide(title, bullets):
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    title_box = slide.shapes.add_textbox(Inches(0.6), Inches(0.35), Inches(12.1), Inches(0.7))
    title_frame = title_box.text_frame
    title_frame.text = title
    title_frame.paragraphs[0].font.size = Pt(30)
    title_frame.paragraphs[0].font.bold = True

    body = slide.shapes.add_textbox(Inches(0.8), Inches(1.35), Inches(11.8), Inches(5.6))
    tf = body.text_frame
    tf.word_wrap = True
    for i, bullet in enumerate(bullets):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = bullet
        p.font.size = Pt(20)
        p.level = 0
    return slide

add_slide('Сегментация клиентов', [
    'Цель: выделить группы клиентов с похожим покупательским поведением.',
    'Источник данных: online_retail_II.csv.',
    'Основа анализа: транзакции, выручка, частота покупок и давность последней покупки.',
    'Бизнес-результат: персонализация маркетинга и повышение удержания клиентов.'
])

add_slide('Данные и предобработка', [
    f'Исходные данные: {df.shape[0]} строк, {df.shape[1]} столбцов.',
    'Удалены строки с пропусками в ключевых полях, отмененные заказы и некорректные значения.',
    'Обработаны выбросы по стоимости строки заказа через IQR.',
    f'После очистки осталось строк: {data_clean.shape[0]}; клиентов в RFM-таблице: {len(rfm)}.'
])

add_slide('RFM-признаки', [
    'Recency: количество дней с последней покупки.',
    'Frequency: число покупок клиента.',
    'Monetary: суммарная выручка по клиенту.',
    'Дополнительно: средний чек, количество товаров и число уникальных позиций.',
    'Масштабирование: StandardScaler.'
])

add_slide('Алгоритмы кластеризации', [
    'K-Means: сегментация на компактные группы.',
    'Иерархическая кластеризация: анализ вложенной структуры сегментов.',
    'DBSCAN и OPTICS: поиск групп произвольной формы и нетипичных клиентов.',
    f'Оптимальное k для K-Means/Hierarchical: {best_k}.'
])

add_slide('Оценка качества', [
    'Использованы внутренние метрики: Silhouette, Davies-Bouldin, Calinski-Harabasz.',
    'ARI и NMI не рассчитаны, потому что истинные метки сегментов отсутствуют.',
    f'Лучший алгоритм по Silhouette: {best_algorithm}.',
    f"Silhouette лучшей модели: {metrics_table.loc[best_algorithm, 'silhouette']:.4f}."
])

segment_lines = [
    f"Кластер {idx}: {row['SegmentName']}; клиентов: {int(row['CustomerCount'])}."
    for idx, row in cluster_profiles.iterrows()
]
add_slide('Полученные сегменты', segment_lines[:6])

add_slide('Бизнес-рекомендации', [
    'Лояльным ценным клиентам: VIP-программа и персональные предложения.',
    'Регулярным покупателям: кросс-продажи и увеличение среднего чека.',
    'Клиентам со сниженной активностью: реактивационные кампании.',
    'Клиентам с крупными покупками: премиальный ассортимент и персональное сопровождение.',
    'Новым клиентам: приветственные предложения и рекомендации популярных товаров.'
])

add_slide('Вывод', [
    'Кластеризация позволяет перейти от массового маркетинга к управлению клиентскими сегментами.',
    'RFM-признаки дают интерпретируемую основу для бизнес-решений.',
    'Результаты можно использовать для удержания клиентов, роста выручки и оптимизации коммуникаций.'
])

prs.save(presentation_path)
print('Презентация сохранена:', presentation_path)

